<h1>Chapter 11 - Coding Agents</h1>
<i>Create an Agent for developing code.</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 11 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

# ▂▂▂▂▂▂▂▂▂▂▂▂

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [ ]:
import os
from illustrated_agents.llm import LLM

# Ollama
llm = LLM(model="ollama/gemma3:12b")

# Llama.cpp server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M", api_base="http://localhost:8080", api_key="sk-no-key-required")
 
# Llama-cpp-python server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M.gguf", api_base="http://localhost:8000/v1/", api_key="sk-no-key-required")

# LM Studio
# llm = LLM(model="lm_studio/gemma-3-12b-it", api_base="http://localhost:1234/v1", api_key="sk-no-key-required")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_GEMINI_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash")
# llm = LLM(model="gemini/gemma-3-12b-it")

## 2 - The Coding Agent

* TODO: Add tools and potentially better parsing


## 3 - The Styling

In [ ]:
import re
from rich.console import Console
from rich.rule import Rule

console = Console()


class Display:
    """Formats agent events for the terminal."""

    def __init__(self):
        self._status = None

    def __call__(self, event, data=None):

        # Animate thinking
        if event == "thinking":
            self._status = console.status("Thinking...", spinner="dots")
            self._status.start()

        # Print THOUGHT
        elif event == "response":
            self.stop()
            thought = re.search(r"THOUGHT:\s*(.+?)(?=ACTION:|$)", data, re.IGNORECASE | re.DOTALL).group(1).strip()
            console.print(f"  [bold dark_orange]{'THOUGHT':<13}[/][dim italic]{thought}[/]\n")

        # Print ACTION
        elif event == "tool_call" and data:
            tool = data.get("tool")
            if tool != "final_answer":
                args = ", ".join(f"{k}={v!r}" for k, v in data.get("kwargs", {}).items())
                console.print(f"  [bold yellow]{'ACTION':<13}[/][yellow]{tool}({args})[/]")

        # Print OBSERVATION
        elif event == "observation":
            console.print(f"  [bold green]{'OBSERVATION':<13}[/]{data}\n")
            console.print(Rule(style="dim"), end="\n\n")

    def stop(self):
        if self._status:
            self._status.stop()
            self._status = None

In [ ]:
from illustrated_agents.chapters.ch11 import display_annotated; display_annotated

In [ ]:
display = Display()
display("response", "THOUGHT: This is a sample thought.\nACTION: search('example query')")
display("tool_call", {"tool": "search", "kwargs": {"query": "example query"}})
display("observation", "This is a sample observation.")
display.stop()

## 4 - The `TinyAgent`

In [ ]:
from illustrated_agents.chapters.ch11 import tinyagents_diff; tinyagents_diff

## 5 - The Chat Interface

To create an interface, we are going to explore a main component, namely the interface with `rich`.

In [ ]:
def chat(agent):
    """Interactive chat loop — works in both terminal and Jupyter."""
    display = agent.display

    # Agent Overview
    console.print(f"\n  [dim]Tools:[/]  {', '.join(agent.tools.tools.keys())}")
    console.print("  [dim]Type[/] exit [dim]to quit.[/]\n")

    while True:
        # User input
        try:
            query = input("> ").strip()
        except (KeyboardInterrupt, EOFError):
            console.print("  [dim]Goodbye![/]")
            break

        # Skip empty input
        if not query:
            continue

        console.print(f"  [bold]> [/]{query}\n")

        # Exit commands
        if query.lower() in ("exit", "quit"):
            console.print("  [dim]Goodbye![/]")
            break

        # Run agent
        try:
            result = agent.run(query)
            display.stop()
            console.print(f"\n  [bold cyan]{'ANSWER':<13}[/]{result}\n")
        except Exception as e:
            display.stop()
            console.print(f"  [bold red]{'ERROR':<13}[/][red]{e}[/]\n")

## 6 - Putting It All Together

In [ ]:
from illustrated_agents import Memory, Tools, ReAct, Reflector, Skills
from illustrated_agents.chapters.ch11 import TinyAgent

# Tools
def calculator(a: str, b: str) -> float:
    return float(a) + float(b)

def get_weather(location: str) -> str:
    return f"Weather in {location}: Sunny, 72F"

tools = Tools()
tools.add_tool("calculator", calculator, "Adds two numbers: calculator(a, b)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(location)")

# Agent
agent = TinyAgent(
    llm=llm, 
    memory=Memory(), 
    tools=tools,
    planner=ReAct(max_steps=10), 
    reflector=Reflector(interval=20),
    skills=Skills(), 
    display=Display(),
)

In [ ]:
# Start interactive chat
chat(agent)

## 7 - The "Real" CLI

TODO: Add description of how the `cli.py` works.